In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

In [2]:
ratings = pd.read_csv('../ml-25m/ratings.csv')

print(ratings.shape)

(25000095, 4)


In [3]:
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_map = {
    old:new
    for new, old in enumerate(user_ids)
}

movie_map = {
    old:new
    for new, old in enumerate(movie_ids)
}

In [4]:
ratings['user_idx'] = ratings['userId'].map(user_map)
ratings['movie_idx'] = ratings['movieId'].map(movie_map)

In [5]:
R = csr_matrix(
    (
        ratings['rating'],
        (
            ratings['user_idx'],
            ratings['movie_idx']
        )
    )
)

print(R.shape)

(162541, 59047)


In [6]:
k = 50

U, sigma, Vt = svds(R, k=k)

sigma_matrix = np.diag(sigma)

In [7]:
target_user = 1

In [8]:
user_idx = user_map[target_user]

print(user_idx)

0


In [9]:
user_predictions = np.dot(
    np.dot(U[user_idx, :], sigma_matrix),
    Vt
)

print(user_predictions.shape)

(59047,)


In [10]:
pred_df = pd.DataFrame({
    "movie_idx": range(len(user_predictions)),
    "predicted_rating": user_predictions
})

pred_df.head()

,movie_idx,predicted_rating
0,0,1.284276
1,1,0.816189
2,2,0.835061
3,3,0.282045
4,4,0.332955


In [11]:
seen_movies = ratings[
    ratings['userId'] == target_user
]['movie_idx'].tolist()

len(seen_movies)

70

In [12]:
pred_df = pred_df[
    ~pred_df['movie_idx'].isin(seen_movies)
]

In [13]:
top10 = pred_df.sort_values(
    by='predicted_rating',
    ascending=False
).head(10)

top10

,movie_idx,predicted_rating
439,439,1.426121
227,227,1.295673
451,451,1.271964
295,295,1.215696
202,202,1.194943
344,344,1.158643
297,297,1.150405
199,199,1.042826
193,193,1.041840
236,236,1.025171


In [14]:
reverse_movie_map = {
    v:k
    for k,v in movie_map.items()
}

In [15]:
top10['movieId'] = top10['movie_idx'].map(
    reverse_movie_map
)

top10

,movie_idx,predicted_rating,movieId
439,439,1.426121,6874
227,227,1.295673,7153
451,451,1.271964,7438
295,295,1.215696,2858
202,202,1.194943,4993
344,344,1.158643,4226
297,297,1.150405,2959
199,199,1.042826,4886
193,193,1.041840,4306
236,236,1.025171,8961


In [16]:
movies = pd.read_csv('../ml-25m/movies.csv')

In [17]:
recommendations = top10.merge(
    movies,
    on='movieId'
)

recommendations[
    ['movieId', 'title', 'genres']
]

,movieId,title,genres
0,6874,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
1,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
2,7438,Kill Bill: Vol. 2 (2004),Action|Drama|Thriller
3,2858,American Beauty (1999),Drama|Romance
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
5,4226,Memento (2000),Mystery|Thriller
6,2959,Fight Club (1999),Action|Crime|Drama|Thriller
7,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
8,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
9,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy


# Day 4 — Top-N Recommendation Generation

## Objective
Generate personalized movie recommendations using the latent factors obtained from Singular Value Decomposition (SVD).

---

## Tasks Completed

### 1. User Selection
Selected a target user from the dataset.

### 2. Rating Prediction
Generated predicted ratings using:

User Prediction = U_user × Σ × Vᵀ

for all movies.

### 3. Recommendation Candidate Generation
Created a list of movies along with predicted ratings.

### 4. Seen Movie Filtering
Removed all movies already rated by the target user.

### 5. Top-N Ranking
Sorted predicted ratings in descending order.

Selected the Top 10 highest-ranked unseen movies.

### 6. Movie Metadata Mapping
Mapped movie indices back to MovieLens movie IDs.

Joined recommendation results with movies.csv.

### 7. Recommendation Display
Displayed movie titles and genres.

---

## Key Findings

1. Successfully generated personalized movie recommendations.
2. Filtered previously watched content.
3. Produced Top-10 unseen movie recommendations.
4. Verified that SVD latent factors can be used for recommendation generation.

---

## Deliverables Produced

- 04_Recommendations.ipynb
- Predicted Rating Scores
- Top 10 Recommendations
- Movie Title Mapping

---

## Status

Day 4: COMPLETED ✅

Next Stage:
Day 5 — RMSE and MAE Evaluation